# 05 · RLHF: moverlo hacia lo que las personas prefieren

**Demo rápida — Deep Learning para Business Analytics · Comillas ICADE**

En el cuaderno 03 le enseñamos al modelo una tarea con ejemplos resueltos. Funciona cuando existe
**una** respuesta correcta y la puedes escribir.

Banco Iberia tiene ahora otro problema. Su manual de atención al cliente (`ATC-01`) dice que toda
respuesta escrita debe reconocer el problema, decir qué va a hacer el banco y dar un plazo, en
segunda persona y sin jerga. Eso describe un **criterio de calidad**, bajo el cual hay mil
respuestas válidas y mil que no lo son, y nadie va a escribir el manual en forma de 160 pares
entrada-salida.

Lo que sí es barato es **comparar**: enséñale a una persona dos respuestas y pregúntale cuál
prefiere. Esa es toda la idea de **RLHF** (*Reinforcement Learning from Human Feedback*), la
técnica que convirtió GPT-3 en ChatGPT.

![Las tres etapas de RLHF](figs/05_rlhf_etapas.png)

Tres etapas. La etapa 1 ya está hecha: SmolLM2-135M-**Instruct** es el resultado de un SFT. Vamos
a hacer la 2 y la 3.

In [ ]:
import sys
sys.path.insert(0, ".")

import random
import time

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from comun.config import (MODELO_PEQUE, MODELO_CHARLA, DTYPE_ENTRENO, DTYPE_CHARLA,
                          DEVICE, fijar_semilla, cronometro, resumen_entorno)
from comun.datos import preferencias, INSTRUCCION_TONO
from comun.entrenamiento import construir_lote, log_prob_respuesta

resumen_entorno()
fijar_semilla()

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODELO_PEQUE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

train, test = preferencias(n_test=5)
print(f"pares de preferencia · entrenamiento: {len(train)}   prueba: {len(test)}\n")
print("--- LA INSTRUCCIÓN ---")
print(INSTRUCCION_TONO)
print("\n--- UN PAR ---")
d = train[0]
print(f"\nPROMPT   : {d['prompt']}")
print(f"\nELEGIDA  : {d['elegida']}")
print(f"\nRECHAZADA: {d['rechazada']}")
print("\nLas dos están bien escritas en castellano. La rechazada incumple el manual:")
print("no reconoce el problema, no dice qué va a hacer el banco y no da plazo.")

## 1. El punto de partida: el modelo prefiere la respuesta mala

Antes de nada, una medida que no suele hacerse y que explica por qué hace falta todo lo demás:
¿a cuál de las dos respuestas le da el modelo base más probabilidad?

In [ ]:
from transformers import AutoModelForCausalLM

with cronometro("cargar el modelo base"):
    base = AutoModelForCausalLM.from_pretrained(MODELO_PEQUE, dtype=DTYPE_ENTRENO).to(DEVICE)
base.eval()


def margen_crudo(datos, modelo):
    """log pi(elegida | x) - log pi(rechazada | x), por par."""
    Xc, Ac, Yc = construir_lote(
        tok, [{"entrada": d["prompt"], "salida": d["elegida"]} for d in datos], INSTRUCCION_TONO)
    Xr, Ar, Yr = construir_lote(
        tok, [{"entrada": d["prompt"], "salida": d["rechazada"]} for d in datos], INSTRUCCION_TONO)
    with torch.no_grad():
        lc = log_prob_respuesta(modelo, Xc.to(DEVICE), Ac.to(DEVICE), Yc.to(DEVICE))
        lr = log_prob_respuesta(modelo, Xr.to(DEVICE), Ar.to(DEVICE), Yr.to(DEVICE))
    return lc - lr


m = margen_crudo(test, base)
print(f"Pares en los que el modelo base prefiere la BUENA: "
      f"{(m > 0).float().mean():.0%}  ({(m > 0).sum().item()}/{len(test)})")
print(f"Margen medio: {m.mean():.1f} nats\n")
print("Un margen de -53 significa que el modelo base considera la respuesta")
print("corporativa unas e^53 veces más probable que la buena. Y es razonable:")
print("se preentrenó con internet, e internet está lleno de prosa corporativa.")
print("Nuestro objetivo es lo que prefiere una persona, y eso hay que enseñárselo aparte.")

## 2. Etapa 2: el modelo de recompensa

Pedirle a una persona que puntúe una respuesta del 1 al 10 no funciona: la escala se mueve entre
anotadores y entre días. Pedirle que **elija entre dos** sí es estable.

El puente entre una elección y un número es el modelo de **Bradley-Terry**, de 1952. Si cada
respuesta tiene una puntuación latente $r_\phi$, la probabilidad de que un humano prefiera $y_b$
sobre $y_m$ es la sigmoide de la diferencia:

$$P(y_b \succ y_m \mid x) = \sigma\!\left(r_\phi(x, y_b) - r_\phi(x, y_m)\right)$$

Maximizar la verosimilitud de las elecciones observadas da la pérdida del modelo de recompensa:

$$\mathcal{L}_{RM} = -\log \sigma\!\left(r_\phi(x, y_b) - r_\phi(x, y_m)\right)$$

Fíjate en que $r_\phi$ nunca se supervisa directamente. Solo se le pide que ordene bien.

![Bradley-Terry](figs/05_bradley_terry.png)

In [ ]:
# El modelo de recompensa es el mismo transformer con una cabeza escalar en vez
# de la cabeza de vocabulario: en lugar de 49.152 logits, devuelve un número.
from transformers import AutoModelForSequenceClassification

fijar_semilla()
rm = AutoModelForSequenceClassification.from_pretrained(
    MODELO_PEQUE, num_labels=1, dtype=DTYPE_ENTRENO).to(DEVICE)
rm.config.pad_token_id = tok.pad_token_id

print(f"Cabeza del modelo de recompensa: {rm.score}")
print(f"Devuelve {rm.num_labels} número por secuencia, sin ninguna escala fijada.")
print("Solo tienen sentido las DIFERENCIAS entre puntuaciones, nunca el valor absoluto.")


def puntuar(prompts, respuestas):
    """r(x, y) para cada par. Shape de salida: (batch,)."""
    textos = [tok.apply_chat_template(
        [{"role": "system", "content": INSTRUCCION_TONO},
         {"role": "user", "content": p},
         {"role": "assistant", "content": r}], tokenize=False)
        for p, r in zip(prompts, respuestas)]
    ent = tok(textos, return_tensors="pt", padding=True, truncation=True,
              max_length=320, add_special_tokens=False).to(DEVICE)
    return rm(**ent).logits.squeeze(-1)


def acierto_rm(datos):
    with torch.no_grad():
        a = puntuar([d["prompt"] for d in datos], [d["elegida"] for d in datos])
        b = puntuar([d["prompt"] for d in datos], [d["rechazada"] for d in datos])
    return (a > b).float().mean().item(), (a - b).mean().item()


acc, mar = acierto_rm(test)
print(f"\nAntes de entrenar: acierto {acc:.0%}, margen {mar:+.2f}  (la cabeza es ruido)")

In [ ]:
# El entrenamiento del modelo de recompensa: una pérdida de una línea.
EPOCAS_RM, LR_RM, BATCH_RM = 3, 1e-5, 2

optimizador = torch.optim.AdamW(rm.parameters(), lr=LR_RM)
perdidas_rm, historial = [], []

t0 = time.time()
for epoca in range(EPOCAS_RM):
    random.shuffle(train)
    rm.train()
    for i in range(0, len(train), BATCH_RM):
        lote = train[i:i + BATCH_RM]
        r_buena = puntuar([d["prompt"] for d in lote], [d["elegida"] for d in lote])
        r_mala = puntuar([d["prompt"] for d in lote], [d["rechazada"] for d in lote])

        perdida = -F.logsigmoid(r_buena - r_mala).mean()      # Bradley-Terry

        perdida.backward()
        torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0)
        optimizador.step()
        optimizador.zero_grad()
        perdidas_rm.append(perdida.item())

    rm.eval()
    acc, mar = acierto_rm(test)
    historial.append((acc, mar))
    print(f"época {epoca + 1}/{EPOCAS_RM}  pérdida {sum(perdidas_rm[-7:]) / 7:.4f}  "
          f"| prueba: acierto {acc:.0%}, margen {mar:+.2f}  [{time.time() - t0:.0f} s]")

del optimizador
for p in rm.parameters():
    p.requires_grad = False

Acierto del 100 % sobre los pares de prueba. Hay que decir inmediatamente lo que vale ese número:
**son cinco pares**. Con cinco ejemplos, un 100 % y un 80 % son indistinguibles. Llama 2 usó cerca
de 1,4 millones de comparaciones humanas para entrenar sus modelos de recompensa, y el dato de
esa etapa es la parte cara del RLHF: el cómputo se abarata solo, las personas que comparan no.

Lo que sí podemos hacer con cinco pares es una cosa mucho más informativa que medir el acierto:
**preguntarle al modelo de recompensa qué es lo que ha aprendido**.

In [ ]:
# Sondas: seis respuestas escritas a propósito para ver qué puntúa el RM.
P = "Me han cobrado 240 euros que no reconozco."
SONDAS = {
    "cumple el manual":
        "Entiendo el susto. Bloqueo la tarjeta ahora y abro la reclamación. "
        "Tendrás el dinero devuelto en 10 días hábiles.",
    "corporativa (la del conjunto de rechazadas)":
        "Le informamos de que su solicitud ha sido registrada y será tramitada "
        "conforme a los procedimientos internos vigentes.",
    "tono correcto pero vacía":
        "Entiendo. Lo hago ahora mismo y te confirmo en 24 horas.",
    "palabrería con las palabras clave":
        "Entiendo bloqueo ahora mismo hoy plazo 24 horas te devuelvo reclamación abro mañana.",
    "tono correcto y desprecio al cliente":
        "Entiendo tu problema y lo soluciono hoy, aunque deberías revisar tus cuentas "
        "tú mismo antes de molestarnos.",
    "una sola palabra":
        "Vale.",
}

with torch.no_grad():
    puntos = puntuar([P] * len(SONDAS), list(SONDAS.values()))

print(f"{'puntuación':>11s}   sonda")
print("-" * 76)
for (etiqueta, _), s in sorted(zip(SONDAS.items(), puntos.tolist()), key=lambda t: -t[1]):
    print(f"{s:11.2f}   {etiqueta}")

Lee el orden. El modelo de recompensa pone arriba la que cumple el manual y abajo la corporativa,
que es lo que le pedimos. Y aguanta dos trampas que no esperábamos que aguantase con catorce
pares: la palabrería que repite las palabras clave y el desprecio con tono amable se van al fondo.

Donde se le ve la costura es en "tono correcto pero vacía", que queda por encima de la corporativa
sin decir absolutamente nada. Lo que ha aprendido este modelo de recompensa es sobre todo el
**registro**: tuteo, frases cortas, verbos en primera persona. El contenido lo juzga mucho peor.

Esto importa porque en la etapa 3 vamos a optimizar contra él. Y un optimizador encuentra siempre
el agujero que tú no viste.

## 3. Etapa 3a: best-of-N, el RLHF que no entrena nada

El uso más barato de un modelo de recompensa: generar N respuestas, puntuarlas todas y servir la
mejor. No se toca un solo peso, se implementa en una tarde y funciona.

Lo que cuesta es inferencia: N veces más cómputo por respuesta. Por eso se usa donde la calidad
vale más que la latencia.

Generamos con Qwen2.5-1.5B, que escribe español de verdad, y puntuamos con el modelo de
recompensa que acabamos de entrenar. Que el juez y el generador sean modelos distintos no es
problema: el juez solo lee texto.

In [ ]:
from transformers import AutoTokenizer as AT

with cronometro("cargar Qwen2.5-1.5B"):
    qtok = AT.from_pretrained(MODELO_CHARLA)
    qmod = AutoModelForCausalLM.from_pretrained(MODELO_CHARLA, dtype=DTYPE_CHARLA,
                                                low_cpu_mem_usage=True)


def candidatos(prompt, n=4, temperatura=1.2, max_new_tokens=70):
    """Genera n respuestas distintas a la misma pregunta."""
    texto = qtok.apply_chat_template(
        [{"role": "system", "content": INSTRUCCION_TONO},
         {"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    ent = qtok([texto] * n, return_tensors="pt", add_special_tokens=False)
    with torch.no_grad():
        out = qmod.generate(**ent, max_new_tokens=max_new_tokens, do_sample=True,
                            temperature=temperatura, top_p=0.95,
                            pad_token_id=qtok.eos_token_id)
    return [qtok.decode(x, skip_special_tokens=True).strip()
            for x in out[:, ent.input_ids.shape[1]:]]


fijar_semilla()
N = 4
for d in test[:2]:
    with cronometro(f"generar y puntuar {N} candidatos"):
        cands = candidatos(d["prompt"], n=N)
        with torch.no_grad():
            s = puntuar([d["prompt"]] * N, cands)
    print(f"\nP: {d['prompt']}")
    for punt, c in sorted(zip(s.tolist(), cands), key=lambda t: -t[0]):
        marca = "  ELEGIDA" if punt == max(s.tolist()) else "         "
        print(f" {punt:7.2f}{marca}  {c[:105]!r}")

del qmod, qtok      # 3 GB fuera de la RAM antes de seguir

Compara la elegida con la última de cada bloque. El modelo de recompensa, entrenado con catorce
comparaciones, ordena de forma razonable respuestas que no ha visto nunca y que vienen de otro
modelo.

Léelo también con escepticismo: entre las mejor puntuadas se cuelan respuestas en *usted*, que el
manual prohíbe. Con catorce pares no da para más.

## 4. Etapa 3b: DPO, la que sí mueve los pesos

Lo clásico aquí es **PPO**: generar, puntuar y empujar la política hacia lo puntuado alto con
*policy gradient*. En 2023, Rafailov et al. demostraron algo que simplificó el campo entero: la
política óptima de ese problema se puede obtener **en una sola pérdida supervisada sobre los pares
de preferencia**, sin modelo de recompensa y sin generar nada durante el entrenamiento.

$$\mathcal{L}_{DPO} = -\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_b \mid x)}{\pi_{ref}(y_b \mid x)} - \beta \log \frac{\pi_\theta(y_m \mid x)}{\pi_{ref}(y_m \mid x)}\right)$$

Es la misma sigmoide de Bradley-Terry. Lo que cambia es quién hace de $r$: ahora la recompensa
**es la propia política**, medida como cuánto se ha separado del modelo de referencia $\pi_{ref}$,
que es una copia congelada del modelo de partida.

Esa división por $\pi_{ref}$ no es cosmética. Es la penalización KL metida dentro del objetivo:
el modelo solo gana si sube la probabilidad de la buena **más de lo que sube la de la mala**, y
alejarse de la referencia sin ganar preferencia le cuesta.

In [ ]:
# La política que se entrena, y la referencia que se queda quieta.
fijar_semilla()
politica = AutoModelForCausalLM.from_pretrained(MODELO_PEQUE, dtype=DTYPE_ENTRENO).to(DEVICE)
referencia = AutoModelForCausalLM.from_pretrained(MODELO_PEQUE, dtype=DTYPE_ENTRENO).to(DEVICE)
referencia.eval()
for p in referencia.parameters():
    p.requires_grad = False

BETA = 0.1


def margen_implicito(datos):
    """La recompensa implícita de DPO: beta * [(logp_pol - logp_ref)_buena - (...)_mala]."""
    Xc, Ac, Yc = construir_lote(
        tok, [{"entrada": d["prompt"], "salida": d["elegida"]} for d in datos], INSTRUCCION_TONO)
    Xr, Ar, Yr = construir_lote(
        tok, [{"entrada": d["prompt"], "salida": d["rechazada"]} for d in datos], INSTRUCCION_TONO)
    with torch.no_grad():
        pc = log_prob_respuesta(politica, Xc.to(DEVICE), Ac.to(DEVICE), Yc.to(DEVICE))
        pr = log_prob_respuesta(politica, Xr.to(DEVICE), Ar.to(DEVICE), Yr.to(DEVICE))
        rc = log_prob_respuesta(referencia, Xc.to(DEVICE), Ac.to(DEVICE), Yc.to(DEVICE))
        rr = log_prob_respuesta(referencia, Xr.to(DEVICE), Ar.to(DEVICE), Yr.to(DEVICE))
    return BETA * ((pc - rc) - (pr - rr))


imp = margen_implicito(test)
print(f"Margen implícito antes de entrenar: {imp.mean():+.3f}")
print("Es exactamente cero, y tiene que serlo: la política ES la referencia.")
print(f"La medida informativa del punto de partida es la del apartado 1: "
      f"{margen_crudo(test, politica).mean():.1f} nats a favor de la mala.")

In [ ]:
# El bucle de DPO. Cuatro pasadas hacia delante por lote: la política sobre la
# buena y la mala (con gradiente) y la referencia sobre las dos (sin gradiente).
# En producción, las de la referencia se calculan UNA vez y se cachean.
EPOCAS_DPO, LR_DPO, BATCH_DPO = 4, 5e-6, 2

optimizador = torch.optim.AdamW(politica.parameters(), lr=LR_DPO)
perdidas_dpo, curva = [], []

t0 = time.time()
for epoca in range(EPOCAS_DPO):
    random.shuffle(train)
    politica.train()
    for i in range(0, len(train), BATCH_DPO):
        lote = train[i:i + BATCH_DPO]
        Xc, Ac, Yc = construir_lote(
            tok, [{"entrada": d["prompt"], "salida": d["elegida"]} for d in lote], INSTRUCCION_TONO)
        Xr, Ar, Yr = construir_lote(
            tok, [{"entrada": d["prompt"], "salida": d["rechazada"]} for d in lote], INSTRUCCION_TONO)
        Xc, Ac, Yc = Xc.to(DEVICE), Ac.to(DEVICE), Yc.to(DEVICE)
        Xr, Ar, Yr = Xr.to(DEVICE), Ar.to(DEVICE), Yr.to(DEVICE)

        with torch.no_grad():
            ref_c = log_prob_respuesta(referencia, Xc, Ac, Yc)
            ref_r = log_prob_respuesta(referencia, Xr, Ar, Yr)
        pol_c = log_prob_respuesta(politica, Xc, Ac, Yc)
        pol_r = log_prob_respuesta(politica, Xr, Ar, Yr)

        logits = BETA * ((pol_c - ref_c) - (pol_r - ref_r))
        perdida = -F.logsigmoid(logits).mean()

        perdida.backward()
        torch.nn.utils.clip_grad_norm_(politica.parameters(), 1.0)
        optimizador.step()
        optimizador.zero_grad()
        perdidas_dpo.append(perdida.item())

    politica.eval()
    imp = margen_implicito(test)
    curva.append((imp.mean().item(), (imp > 0).float().mean().item()))
    print(f"época {epoca + 1}/{EPOCAS_DPO}  pérdida {sum(perdidas_dpo[-7:]) / 7:.4f}  "
          f"| prueba: margen implícito {imp.mean():+.3f}, acierto {(imp > 0).float().mean():.0%}  "
          f"[{time.time() - t0:.0f} s]")

del optimizador

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(perdidas_dpo, color="#D9C28A", lw=1)
suave = [sum(perdidas_dpo[max(0, i - 4):i + 1]) / len(perdidas_dpo[max(0, i - 4):i + 1])
         for i in range(len(perdidas_dpo))]
axes[0].plot(suave, color="#B8860B", lw=2.4)
axes[0].axhline(-torch.log(torch.sigmoid(torch.tensor(0.0))).item(), color="#8A8A8A",
                ls=":", lw=1)
axes[0].text(0.5, 0.70, "log 2: el valor cuando no\nse prefiere ninguna", fontsize=8,
             color="#8A8A8A", transform=axes[0].transData)
axes[0].set_xlabel("paso"); axes[0].set_ylabel("pérdida DPO")
axes[0].set_title("La pérdida", fontsize=11, fontweight="bold")

ep = range(1, len(curva) + 1)
axes[1].plot(ep, [c[0] for c in curva], "o-", color="#B8860B", lw=2.4, ms=7)
axes[1].axhline(0, color="#8A8A8A", ls=":", lw=1)
axes[1].set_xlabel("época"); axes[1].set_ylabel("margen implícito en prueba")
axes[1].set_title("Lo que se mueve en los pares que NO ha visto", fontsize=11, fontweight="bold")
axes[1].set_xticks(list(ep))
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

m_fin = margen_crudo(test, politica)
print(f"log pi(buena) - log pi(mala) en prueba:")
print(f"   antes de DPO : {margen_crudo(test, referencia).mean():+8.1f} nats")
print(f"   después      : {m_fin.mean():+8.1f} nats")
print(f"   pares con la buena por delante: {(m_fin > 0).float().mean():.0%}")

Hay que leer los dos números con cuidado, porque dicen cosas distintas.

El **margen implícito** sube de 0 a algo más de +1,5 y es positivo en los cinco pares que el
modelo no había visto. Eso es exactamente lo que DPO optimiza y significa que la política se ha
movido en la dirección correcta: respecto a la referencia, ha subido la probabilidad de la buena
más de lo que ha subido la de la mala, en pares nuevos.

El **margen crudo**, en cambio, pasa de unos −53 nats a unos −37. Se ha recortado un tercio de la
distancia, pero sigue siendo negativo: en términos absolutos el modelo todavía considera más
probable la respuesta corporativa. Con catorce comparaciones y cuatro épocas no da para más, y
decir "el modelo ya prefiere la buena" sería falso.

Esa diferencia entre las dos medidas no es un detalle técnico. El RLHF no reescribe lo que el
modelo aprendió de internet: lo **inclina**. Cuánto lo inclina depende de cuántas comparaciones le
des, y es la razón por la que los laboratorios recogen cientos de miles.

Lo que **no** vamos a hacer es enseñar textos generados por esta política y decir que son mejores.
SmolLM2-135M no escribe español coherente, así que generar con él y juzgar el resultado a ojo no
mediría el efecto de DPO: mediría el ruido de un modelo diminuto. El margen implícito en pares no
vistos sí mide lo que DPO cambia, y es la métrica que se reporta en los artículos del método.

Si quieres ver el efecto en texto generado, el ejercicio 5.5 lo repite sobre Qwen2.5-1.5B, que
pide GPU pero cuyas salidas sí se pueden leer.

## 5. Etapa 3c: el bucle de policy gradient, y por qué lleva el freno KL

DPO se salta la generación. El RLHF clásico no: genera, puntúa y empuja. El objetivo es

$$\max_\theta \; \mathbb{E}_{y \sim \pi_\theta}\big[r_\phi(x, y)\big] \;-\; \beta\,\mathrm{KL}\!\left(\pi_\theta \,\|\, \pi_{ref}\right)$$

El primer término dice "haz lo que el modelo de recompensa puntúa alto". El segundo, "sin alejarte
de lo que eras". PPO, GRPO y los demás son formas distintas de estimar el gradiente del primer
término con poca varianza; el segundo está en todos.

Vamos a programar la versión más simple, REINFORCE con una línea base, y a ejecutarla **con
$\beta = 0$** para ver qué pasa cuando quitas el freno.

![Por qué el freno KL](figs/05_kl.png)

In [ ]:
# Esta celda tarda unos 135 segundos en CPU. Es la más lenta de la demo.
PROMPTS = [d["prompt"] for d in train]
BETA_KL = 0.0          # <- el freno, desactivado a propósito. Pruébalo con 0.2.
PASOS, LOTE, LR_PG, N_TOK = 20, 6, 2e-5, 48


def muestrear(modelo, prompts, max_new_tokens=N_TOK, temperatura=1.0):
    """Genera una respuesta por prompt. Padding por la izquierda, que es lo que
    exige generate cuando el lote tiene longitudes distintas."""
    tok.padding_side = "left"
    textos = [tok.apply_chat_template(
        [{"role": "system", "content": INSTRUCCION_TONO}, {"role": "user", "content": p}],
        add_generation_prompt=True, tokenize=False) for p in prompts]
    ent = tok(textos, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
    with torch.no_grad():
        out = modelo.generate(**ent, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=temperatura, top_p=0.95,
                              pad_token_id=tok.pad_token_id)
    tok.padding_side = "right"
    return [tok.decode(x, skip_special_tokens=True).strip()
            for x in out[:, ent["input_ids"].shape[1]:]]

In [ ]:
fijar_semilla()
pg = AutoModelForCausalLM.from_pretrained(MODELO_PEQUE, dtype=DTYPE_ENTRENO).to(DEVICE)
optimizador = torch.optim.AdamW(pg.parameters(), lr=LR_PG)

historia = []
t0 = time.time()
for paso in range(PASOS):
    prompts = random.sample(PROMPTS, LOTE)

    # 1. La política genera. Sin gradiente: esto es recoger experiencia.
    pg.eval()
    respuestas = muestrear(pg, prompts)
    pg.train()

    # 2. El modelo de recompensa puntúa lo generado.
    with torch.no_grad():
        r = puntuar(prompts, respuestas)

    # 3. log pi(y|x) de lo generado, ahora SÍ con gradiente.
    X, A, Y = construir_lote(
        tok, [{"entrada": p, "salida": s} for p, s in zip(prompts, respuestas)], INSTRUCCION_TONO)
    X, A, Y = X.to(DEVICE), A.to(DEVICE), Y.to(DEVICE)
    with torch.no_grad():
        logp_ref = log_prob_respuesta(referencia, X, A, Y)
    logp = log_prob_respuesta(pg, X, A, Y)

    # 4. REINFORCE con línea base: la ventaja es la recompensa menos la media del
    #    lote. Restar una constante no sesga el gradiente y le quita mucha varianza.
    ventaja = (r - r.mean()).detach()
    kl = logp - logp_ref
    perdida = (-(ventaja * logp).mean() + BETA_KL * kl.mean()) / N_TOK

    perdida.backward()
    torch.nn.utils.clip_grad_norm_(pg.parameters(), 1.0)
    optimizador.step()
    optimizador.zero_grad()

    historia.append((r.mean().item(), kl.mean().item()))
    if (paso + 1) % 4 == 0 or paso == 0:
        print(f"paso {paso + 1:2d}/{PASOS}   recompensa {r.mean():+6.2f}   "
              f"KL respecto a la referencia {kl.mean():7.2f}   [{time.time() - t0:.0f} s]")

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4.4))
pasos = range(1, len(historia) + 1)
ax1.plot(pasos, [h[0] for h in historia], "o-", color="#B8860B", lw=2.2, ms=5,
         label="recompensa media según el RM")
ax1.set_xlabel("paso de optimización"); ax1.set_ylabel("recompensa", color="#B8860B")
ax1.axhline(historia[0][0], color="#D9C28A", ls=":", lw=1.2)

ax2 = ax1.twinx()
ax2.plot(pasos, [h[1] for h in historia], "s--", color="#8A8A8A", lw=2,
         ms=4, label="KL respecto a la referencia")
ax2.set_ylabel("KL (nats por respuesta)", color="#8A8A8A")
ax2.spines["top"].set_visible(False); ax1.spines["top"].set_visible(False)
fig.suptitle(f"REINFORCE con beta = {BETA_KL}: la recompensa sube y el modelo se aleja",
             fontsize=12, fontweight="bold")
fig.legend(loc="lower right", bbox_to_anchor=(0.9, 0.18), fontsize=8.5, frameon=False)
plt.tight_layout(); plt.show()

k = 5
r_ini = sum(h[0] for h in historia[:k]) / k
r_fin = sum(h[0] for h in historia[-k:]) / k
kl_fin = sum(h[1] for h in historia[-k:]) / k
print(f"recompensa media, primeros {k} pasos : {r_ini:+.2f}")
print(f"recompensa media, últimos  {k} pasos : {r_fin:+.2f}   ({r_fin - r_ini:+.2f})")
print(f"KL media, últimos {k} pasos          : {kl_fin:.2f} nats")

print("\nY esto es lo que genera ahora la política:\n")
pg.eval()
for p, s in zip(PROMPTS[:3], muestrear(pg, PROMPTS[:3])):
    print(f"  P: {p[:66]}")
    print(f"  R: {s[:120]!r}\n")

Tres cosas en esas dos celdas, y conviene separarlas por cuánto te puedes fiar de cada una.

**La divergencia KL sube sin discusión**, de 0 a varios nats por respuesta. La política se ha
alejado del modelo de partida, y lo ha hecho en veinte pasos.

**El texto generado se ha degradado** hasta fragmentos sin sentido. Compáralos con lo que SmolLM2
escribía al principio del cuaderno: era malo, pero era español.

**La recompensa media sube poco y con mucho ruido.** Con lotes de 6 y 20 pasos, la varianza del
estimador REINFORCE se come casi toda la señal; verás una mejora de unas décimas y curvas que
suben y bajan. No saques conclusiones de la pendiente: saca la conclusión de la combinación de las
tres cosas.

Porque la combinación tiene nombre: **reward hacking**. El optimizador se está moviendo hacia la
región del espacio de textos donde el modelo de recompensa se equivoca, no hacia la región donde
las respuestas son buenas. Está haciendo exactamente lo que le pedimos,
que es maximizar $r_\phi$, y $r_\phi$ nunca fue la calidad real: era un clasificador de registro
entrenado con catorce ejemplos.

La divergencia KL es el detector barato de esto. Cuando sube deprisa mientras la calidad real no
mejora, estás *hackeando* la recompensa. Por eso **el término $-\beta\,\mathrm{KL}$ está en todos
los algoritmos de la familia**, y por eso DPO lo lleva metido en la propia pérdida a través de la
división por $\pi_{ref}$.

Un RLHF de verdad son decenas de miles de pasos con lotes de cientos, y ahí la curva de recompensa
sí es legible. Lo que no cambia con la escala es el fenómeno: cuanto mejor es tu optimizador, más
deprisa encuentra el agujero de tu modelo de recompensa.

**Pon `BETA_KL = 0.2`, vuelve a ejecutar las dos celdas anteriores y compara las dos curvas.**
Es el ejercicio más importante de este cuaderno.

## 6. Qué elegir en la práctica

| | Qué necesita | Qué cuesta | Cuándo |
|---|---|---|---|
| **Best-of-N** | un modelo de recompensa | N× inferencia, cero entrenamiento | para empezar; funciona el primer día |
| **DPO** | pares de preferencia | un entrenamiento supervisado | lo habitual hoy: estable, barato, sin generar |
| **PPO / GRPO** | modelo de recompensa + bucle de generación | caro e inestable | cuando la recompensa es verificable (código, matemáticas) o hay mucho presupuesto |

El criterio corto: **empieza por DPO**. Es más estable, no necesita modelo de recompensa separado
y consume una fracción del cómputo. Se pasa a PPO cuando la recompensa se puede calcular de forma
programática y fiable, que es justo el caso de los modelos de razonamiento recientes: ahí la
recompensa es "¿compila el código?" o "¿da el resultado correcto?", y no hay nada que *hackear*.

## 7. Ejercicios

**5.1** El del apartado 5: `BETA_KL = 0.2`. Dibuja las dos curvas de recompensa y las dos de KL en
la misma figura. ¿A qué paso empiezan a separarse? ¿Qué le pasa al texto generado?

**5.2** Barre `BETA` de DPO en `{0.01, 0.1, 0.5, 1.0}`. Mide el margen implícito en prueba y la
divergencia KL final respecto a la referencia. Explica el compromiso que controla ese parámetro.

**5.3** Rompe el modelo de recompensa a propósito. Escribe diez respuestas diseñadas para
puntuar alto sin ser buenas y mide cuántas lo consiguen. Después añade esas diez al conjunto de
entrenamiento como respuestas rechazadas, reentrena y vuelve a medir. Acabas de hacer, a escala de
juguete, el ciclo de *red teaming* de un laboratorio de IA.

**5.4** Amplía los pares de preferencia de 19 a 60, escribiéndolos tú. Mide el acierto del modelo
de recompensa con 10, 20, 40 y 60 pares de entrenamiento y dibuja la curva. Extrapola cuántos
harían falta para llegar al 95 %, y compara esa extrapolación con el 1,4 millones de Llama 2.

**5.5** Repite el DPO del apartado 4 sobre `Qwen/Qwen2.5-1.5B-Instruct` con LoRA (cuaderno 04) en
lugar del modelo completo: es la combinación que se usa de verdad, porque hay que tener en memoria
la política y la referencia a la vez. Genera antes y después y juzga el texto a ojo, que ahora sí
se puede leer.

**5.6** Implementa PPO en lugar de REINFORCE: añade el ratio $\pi_\theta/\pi_{old}$, el recorte a
$[1-\epsilon, 1+\epsilon]$ y varias épocas de optimización sobre el mismo lote de experiencia.
Mide la fracción de recortes (*clip fraction*) y la KL aproximada de Schulman por iteración.
Compara la estabilidad con la de REINFORCE.